# 03 Connectivity QC and export

Summarize written connectivity files and inspect basic matrix properties.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == "4_connectivity":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)

from meeg_pipeline.config import load_config
from meeg_pipeline.workflow import iter_recordings, recordings_to_dataframe

config = load_config(CONFIG_PATH)
recordings = list(iter_recordings(config, subjects="all"))
recordings_to_dataframe(recordings)

In [ ]:
from meeg_pipeline.connectivity import connectivity_qc_to_dataframe

qc = connectivity_qc_to_dataframe(config)
qc

In [ ]:
# Show sizes and shapes.
if len(qc):
    qc[["size_mb", "shape", "n_labels", "bands", "method", "window", "condition", "n_epochs", "sfreq", "path"]].sort_values("size_mb", ascending=False)
else:
    print("No connectivity files found yet.")

In [ ]:
# Load one connectivity matrix and summarize finite values.
from pathlib import Path
import numpy as np

if len(qc):
    path = Path(qc.iloc[0]["path"])
    with np.load(path, allow_pickle=True) as npz:
        con = npz["connectivity"]
        labels = npz["labels"]
        print("path:", path)
        print("connectivity shape:", con.shape)
        print("n labels:", len(labels))
        print("bands:", npz["band_names"])
        print("finite min/mean/max:", np.nanmin(con), np.nanmean(con), np.nanmax(con))
else:
    print("No connectivity files found yet.")

In [ ]:
# Optional: export QC table.
export_path = config.paths.derivatives_root / "connectivity_qc.tsv"
if len(qc):
    qc.to_csv(export_path, sep="\t", index=False)
    print("Wrote", export_path)
else:
    print("Nothing to export.")